In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [88]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')

In [89]:
# Time
start = dt.datetime(2019,6,25)
end = dt.datetime(2019,6,3)
print(start,end)

2019-06-25 00:00:00 2019-06-03 00:00:00


In [90]:
aw = []
c_users = cursor.superstars.users
for documents in c_users.find({"_id":ObjectId("5d0d1deafa80e3002820af05")}): 
   aw.append(documents)


dic_flattened = [flatten(d) for d in aw]
users = pd.DataFrame(dic_flattened)
aw

NameError: name 'ObjectId' is not defined

In [81]:
users[users['user_id'] == '5d0d1deafa80e3002820af05']

,user_id,create_time,device_id


In [12]:
users.sort_values(['device_id','create_time'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))
users.head()

26398


,user_id,create_time,device_id
23549,5cf94e14fb3482380005cc3c,2019-06-06 17:32:04.453,ffffc06e7d82cd51683d09a7f3dee2a4
8113,5cd72e3de839226f40a38464,2019-05-11 20:19:09.199,ffff9c0b7fd52ade11feb883aba4134e
7797,5cd708cb112ed13abd410d3f,2019-05-11 17:39:23.137,fffeb1875ef302640a96261a39f30aed
7984,5cd71bd5cfd59d105c9a7213,2019-05-11 19:00:37.963,fff782ca29fe2744d971165f49e885cc
21001,5cf09b210721db1fde85b11a,2019-05-31 03:10:25.629,fff76c9ea9dba2d1721492c6819b8626


In [78]:
team_cursor = cursor.superstars.teams
aw_team = []
for documents in team_cursor.find({},{"user":1,'created_at':1,'name':1}): 
    aw_team.append(documents)
dic_flattened = [flatten(d) for d in aw_team]
teams = pd.DataFrame(dic_flattened)
#teams = teams[teams["user"].isin(users["user_id"])]
teams = teams[["_id","user",'created_at','name']]
teams.columns = ["team_id", "user_id","team_create_time","team_name"]

In [80]:
teams = teams[teams['team_name']=='Dynamic Thunders']
teams

,team_id,user_id,team_create_time,team_name
5026,5ca730af74aaef18f702cdfa,5ca730af74aaef18f702cdd5,2019-04-05 10:40:47.905,Dynamic Thunders
5375,5ca7399fc71a3718d610043f,5ca7399fc71a3718d610041a,2019-04-05 11:18:55.787,Dynamic Thunders
5558,5ca73e9a74aaef18f702e648,5ca73e9a74aaef18f702e623,2019-04-05 11:40:10.132,Dynamic Thunders
6896,5ca762d174aaef18f7031d80,5ca762d174aaef18f7031d5b,2019-04-05 14:14:41.616,Dynamic Thunders
8286,5ca782a3d1d73c10740f3c17,5ca782a3d1d73c10740f3bf2,2019-04-05 16:30:27.495,Dynamic Thunders
9241,5ca7a1c7c8ed5426865077e9,5ca7a1c7c8ed5426865077c4,2019-04-05 18:43:19.331,Dynamic Thunders
11934,5ca85e5e4151d11807139a30,5ca85e5e4151d11807139a0b,2019-04-06 08:07:58.830,Dynamic Thunders
13230,5ca87301786243325020f5a2,5ca87300786243325020f57d,2019-04-06 09:36:01.071,Dynamic Thunders
15480,5ca8a47445bac635cc299e49,5ca8a47445bac635cc299e24,2019-04-06 13:07:00.701,Dynamic Thunders
16781,5ca8c2060c1ce74a2f8486d3,5ca8c2060c1ce74a2f8486ae,2019-04-06 15:13:10.756,Dynamic Thunders


In [73]:
c_cards = cursor.superstars.user_collectables_logs # this query will give me how many speedup_cards have been used. 
                                                    #Each tap will be counted as one use of the speedup card
aw_cards = []
for documents in c_cards.aggregate([{'$unwind':"$data"}, 
                                    {"$match" : {'data.quantity': {'$lt': 0},
                                                 'type':'HARD_CURRENCY',
                                               
                                                 'data.created_at': {'$gte': start}}}]):
    aw_cards.append(documents)
 
dic_flattened = [flatten(d) for d in aw_cards]
cards = pd.DataFrame(dic_flattened)
# cards = cards[cards["user"].isin(users_team_player_trained['user_id'])]   # total_trained is the number of times each user has trained
# # cards = cards[['user',"data_created_at",'data_catalogue_id']]
# # cards.columns = ['user_id',"event_timestamp",'value']

In [74]:
cards.sort_values('data_quantity',ascending=True,inplace=True)
cards = cards[cards['data_reason_type']=='KIT_KEYS_PURCHASE']
cards.head()

,__v,_id,data__id,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user
361,0,5d0d2028fa80e30028215807,5d1211c2fe662b0012acd963,1,2019-06-25 12:21:22.305,-120,KIT_KEYS_PURCHASE,HARD_CURRENCY,5d0d1deafa80e3002820af05
360,0,5d0d2028fa80e30028215807,5d1211b751a95c001993d949,1,2019-06-25 12:21:11.831,-120,KIT_KEYS_PURCHASE,HARD_CURRENCY,5d0d1deafa80e3002820af05
358,0,5d0d2028fa80e30028215807,5d12119a51a95c001993cd5a,1,2019-06-25 12:20:42.489,-120,KIT_KEYS_PURCHASE,HARD_CURRENCY,5d0d1deafa80e3002820af05
355,0,5d0d2028fa80e30028215807,5d12117751a95c001993af54,1,2019-06-25 12:20:07.542,-120,KIT_KEYS_PURCHASE,HARD_CURRENCY,5d0d1deafa80e3002820af05
356,0,5d0d2028fa80e30028215807,5d12118351a95c001993b956,1,2019-06-25 12:20:19.151,-120,KIT_KEYS_PURCHASE,HARD_CURRENCY,5d0d1deafa80e3002820af05


In [75]:
a = cards.groupby('user').agg({'data_quantity':'sum'})

In [76]:
a.sort_values('data_quantity',inplace=True)

In [77]:
a

,data_quantity
user,
5d0d1deafa80e3002820af05,-13680
5d11b19afa80e30028e91f8f,-630
5d1188768185c20019d97362,-300
5d131e38114e23001a7aadc9,-120
5d10354ffa80e30028aac20b,-30
5d107a938185c20019accd9d,-30
5d10a7b6fa80e30028be6b40,-30
5d10e7178185c20019c0af35,-30
5d13126b114e23001a77edd5,-30
